In [ ]:
import os
import json
import time
from dotenv import load_dotenv
from openai import OpenAI
from pinecone import Pinecone
from pinecone_text.sparse import BM25Encoder
from pydantic import BaseModel
from typing import Literal

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY')

openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

print("✓ Environment loaded")

In [ ]:
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("financial-reconciliation")

with open("../data/processed/docstore.json", "r") as f:
    docstore = json.load(f)

bm25_encoder = BM25Encoder()
bm25_encoder.load("../data/processed/bm25_encoder.json")

print(f"✓ Pinecone connected")
print(f"✓ Docstore: {len(docstore)} parents")
print(f"✓ BM25 loaded")
print(f"\nIndex stats:")
print(index.describe_index_stats())

In [ ]:
EMBEDDING_MODEL = "text-embedding-3-small"
NOISE_SECTIONS = ["General", "Signature", "Exhibits"]


def get_embeddings(texts: list) -> list:
    response = openai_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts
    )
    return [e.embedding for e in response.data]


def scale_dense(vector: list, alpha: float) -> list:
    return [v * alpha for v in vector]


def scale_sparse(vector: dict, alpha: float) -> dict:
    return {
        "indices": vector["indices"],
        "values": [v * (1 - alpha) for v in vector["values"]]
    }


class QueryRoute(BaseModel):
    chunk_type: Literal["table", "prose"]
    section: Literal[
        "Income Statement", "Balance Sheet", "Cash Flow",
        "MD&A", "Risk Factors", "Legal Proceedings",
        "Notes", "Shareholders Equity", "Any"
    ]


def route_query(query: str) -> dict:
    system_prompt = """You are a financial document query router for SEC 10-Q filings.

PRIORITY RULES:
- inventory/inventories → Balance Sheet, table
- dividends PAID → Cash Flow, table
- R&D as percentage → MD&A, prose
- EPS computation → Notes, table
- gross margin as dollar figure → Income Statement, table
- R&D dollar amount → Income Statement, table
- change/grew/increased about a metric → MD&A, prose

SECTION GUIDE:
- Income Statement: revenue, net sales, gross margin, operating income, net income, cost of sales
- Balance Sheet: total assets, cash, liabilities, equity, receivables, inventory
- Cash Flow: operating activities, investing, financing, dividends paid, capex
- MD&A: why metrics changed, percentage change, growth reasons, segment performance
- Risk Factors: risks, uncertainties, challenges
- Legal Proceedings: lawsuits, investigations
- Notes: RSUs, debt details, EPS computation, weighted average shares
- Shareholders Equity: retained earnings, AOCI
- Any: spans multiple sections

CHUNK TYPE: NUMBER/FIGURE → table | EXPLANATION/WHY/HOW → prose"""

    response = openai_client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": query}
        ],
        response_format=QueryRoute,
        temperature=0
    )
    route = response.choices[0].message.parsed
    return {"chunk_type": route.chunk_type, "section": route.section}


def retrieve_with_parent_child(
    query: str,
    ticker: str,
    section: str = None,
    chunk_type: str = None,
    top_k: int = 5,
    alpha: float = 0.5
) -> list:
    namespace = f"{ticker}_PC"
    dense_vector = get_embeddings([query])[0]
    sparse_vector = bm25_encoder.encode_queries(query)

    filter_dict = {"section": {"$nin": NOISE_SECTIONS}}
    if section and section != "Any":
        filter_dict["section"] = {"$eq": section}
    if chunk_type:
        filter_dict["chunk_type"] = {"$eq": chunk_type}

    results = index.query(
        vector=scale_dense(dense_vector, alpha),
        sparse_vector=scale_sparse(sparse_vector, alpha),
        top_k=top_k * 3,
        namespace=namespace,
        include_metadata=True,
        filter=filter_dict
    )

    seen_parents = set()
    retrieved = []

    for match in results.matches:
        parent_id = match.metadata.get("parent_id")
        if not parent_id or parent_id in seen_parents:
            continue
        seen_parents.add(parent_id)
        parent = docstore.get(parent_id)
        if parent:
            retrieved.append({
                "child_score": match.score,
                "child_content": match.metadata.get("content"),
                "parent_content": parent["content"],
                "metadata": {**match.metadata, "parent_id": parent_id}
            })
        if len(retrieved) >= top_k:
            break

    return retrieved


def retrieve_with_router(query: str, ticker: str, top_k: int = 5, alpha: float = 0.5):
    route = route_query(query)
    retrieved = retrieve_with_parent_child(
        query=query,
        ticker=ticker,
        section=route["section"],
        chunk_type=route["chunk_type"],
        top_k=top_k,
        alpha=alpha
    )
    return retrieved, route


# Quick test
results, route = retrieve_with_router("What was Apple total net sales in Q2 2026?", "AAPL")
print(f"Route: {route}")
print(f"Results: {len(results)}")
print(f"Top result: {results[0]['child_content'][:1000] if results else 'None'}")

In [ ]:
# ragas_compat.py — import this before anything that imports ragas
import sys, types
from langchain_google_vertexai import ChatVertexAI

_shim = types.ModuleType("langchain_community.chat_models.vertexai")
_shim.ChatVertexAI = ChatVertexAI
sys.modules["langchain_community.chat_models.vertexai"] = _shim

In [32]:
from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextPrecision
from ragas.llms import llm_factory
import openai as openai_lib

# Setup using modern RAGAS 0.4.x API
ragas_openai_client = openai_lib.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
ragas_llm = llm_factory("gpt-4o-mini", client=ragas_openai_client)

# Instantiate metrics
faithfulness_metric = Faithfulness(llm=ragas_llm)
response_relevancy_metric = ResponseRelevancy(llm=ragas_llm)
context_recall_metric = LLMContextRecall(llm=ragas_llm)
factual_correctness_metric = FactualCorrectness(llm=ragas_llm)
context_precision_metric = ContextPrecision(llm=ragas_llm)

metrics = [
    faithfulness_metric,
    response_relevancy_metric,
    context_recall_metric,
    factual_correctness_metric,
    context_precision_metric
]

print("✓ RAGAS 0.4.x setup complete")
print("✓ LLM: gpt-4o-mini")
print(f"✓ Metrics: {[m.__class__.__name__ for m in metrics]}")

C:\Users\aksaj\AppData\Local\Temp\ipykernel_22812\2326948638.py:2: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextPrecision
C:\Users\aksaj\AppData\Local\Temp\ipykernel_22812\2326948638.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextPrecision
C:\Users\aksaj\AppData\Local\Temp\ipykernel_22812\2326948638.py:2: DeprecationWarning: Importing FactualCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'raga

✓ RAGAS 0.4.x setup complete
✓ LLM: gpt-4o-mini
✓ Metrics: ['Faithfulness', 'ResponseRelevancy', 'LLMContextRecall', 'FactualCorrectness', 'ContextPrecision']


In [33]:
# System prompt for answer generation
SYSTEM_PROMPT = """You are a financial analyst assistant specializing in SEC filings.
Answer questions based ONLY on the provided financial document context.
Rules:
- State numbers exactly as they appear in the document
- Financial figures in 10-Q tables are in millions unless stated otherwise
- Always mention the exact period: "Three Months Ended March 28, 2026"
- 10-Q tables show BOTH quarterly (Three Months Ended) AND year-to-date (Six Months Ended)
- When asked about a specific quarter → use Three Months Ended column ONLY
- Always append "million" to dollar figures from financial statements
- If you cannot find the answer say: "Not found in the provided filing sections"
- Never fabricate numbers"""


def generate_answer(query: str, ticker: str) -> dict:
    """Run full pipeline and return answer + contexts for RAGAS evaluation."""
    retrieved, route = retrieve_with_router(query, ticker, top_k=3)

    contexts = [r["parent_content"] for r in retrieved]

    if not contexts:
        return {
            "answer": "Not found in the provided filing sections",
            "contexts": [],
            "route": route
        }

    context_text = "\n\n---\n\n".join([
        f"[{r['metadata'].get('section')} | Page {r['metadata'].get('page')}]\n{r['parent_content']}"
        for r in retrieved
    ])

    user_prompt = f"Context:\n{context_text}\n\nQuestion: {query}\n\nAnswer:"

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0
    )

    return {
        "answer": response.choices[0].message.content,
        "contexts": contexts,
        "route": route
    }


# RAGAS evaluation queries with ground truth
EVAL_QUERIES = [
    {
        "question": "What was Apple total net sales in Q2 2026?",
        "ticker": "AAPL",
        "ground_truth": "Apple's total net sales for the Three Months Ended March 28, 2026 were $111,184 million."
    },
    {
        "question": "What was Apple net income in Q2 2026?",
        "ticker": "AAPL",
        "ground_truth": "Apple's net income for the Three Months Ended March 28, 2026 was $29,578 million."
    },
    {
        "question": "What was Apple gross margin in Q2 2026?",
        "ticker": "AAPL",
        "ground_truth": "Apple's gross margin for the Three Months Ended March 28, 2026 was $54,781 million."
    },
    {
        "question": "What was Apple cash and cash equivalents as of March 28 2026?",
        "ticker": "AAPL",
        "ground_truth": "Apple's cash and cash equivalents as of March 28, 2026 were $45,572 million."
    },
    {
        "question": "What was Apple operating income in Q2 2026?",
        "ticker": "AAPL",
        "ground_truth": "Apple's operating income for the Three Months Ended March 28, 2026 was $35,885 million."
    },
    {
        "question": "What was Microsoft net income in Q1 FY2026?",
        "ticker": "MSFT",
        "ground_truth": "Microsoft's net income for the Three Months Ended December 31, 2025 was $24,108 million."
    },
    {
        "question": "What was Microsoft total revenue in Q1 FY2026?",
        "ticker": "MSFT",
        "ground_truth": "Microsoft's total revenue for the Three Months Ended December 31, 2025 was $69,632 million."
    },
    {
        "question": "What was Apple research and development expense in Q2 2026?",
        "ticker": "AAPL",
        "ground_truth": "Apple's research and development expense for the Three Months Ended March 28, 2026 was $11,419 million."
    },
]

print(f"Evaluation queries: {len(EVAL_QUERIES)}")
print("\nGenerating answers...")

samples = []
for i, eq in enumerate(EVAL_QUERIES):
    print(f"  [{i+1}/{len(EVAL_QUERIES)}] {eq['question'][:60]}...")
    result = generate_answer(eq["question"], eq["ticker"])
    samples.append({
        "user_input": eq["question"],
        "response": result["answer"],
        "retrieved_contexts": result["contexts"],
        "reference": eq["ground_truth"]
    })
    print(f"    Answer: {result['answer'][:80]}")

print(f"\n✓ {len(samples)} samples generated")

Evaluation queries: 8

Generating answers...
  [1/8] What was Apple total net sales in Q2 2026?...
    Answer: Apple's total net sales in Q2 2026 (Three Months Ended March 28, 2026) were $ 11
  [2/8] What was Apple net income in Q2 2026?...
    Answer: Apple's net income in the Three Months Ended March 28, 2026 was $ 29,578 million
  [3/8] What was Apple gross margin in Q2 2026?...
    Answer: Apple's gross margin in the Three Months Ended March 28, 2026 was $ 54,781 milli
  [4/8] What was Apple cash and cash equivalents as of March 28 2026...
    Answer: Apple cash and cash equivalents as of March 28, 2026 was $ 45,572 million.
  [5/8] What was Apple operating income in Q2 2026?...
    Answer: Apple's operating income for the Three Months Ended March 28, 2026 was $ 35,885 
  [6/8] What was Microsoft net income in Q1 FY2026?...
    Answer: Not found in the provided filing sections.
  [7/8] What was Microsoft total revenue in Q1 FY2026?...
    Answer: Not found in the provided filing se

In [ ]:
import pandas as pd

# Build dataset using pandas (most stable approach)
data = {
    "user_input": [s["user_input"] for s in samples],
    "response": [s["response"] for s in samples],
    "retrieved_contexts": [s["retrieved_contexts"] for s in samples],
    "reference": [s["reference"] for s in samples]
}

df = pd.DataFrame(data)
eval_dataset = EvaluationDataset.from_pandas(df)

print(f"✓ Dataset built: {len(df)} samples")
print("\nRunning RAGAS evaluation (2-3 minutes)...")

results = evaluate(
    dataset=eval_dataset,
    metrics=metrics
)

print("\n" + "="*50)
print("RAGAS EVALUATION RESULTS")
print("="*50)
print(results)
df_results = results.to_pandas()
# Check actual column names


✓ Dataset built: 8 samples

Running RAGAS evaluation (2-3 minutes)...


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Evaluating:   2%|▎         | 1/40 [03:30<2:16:40, 210.27s/it]Exception raised in Job[3]: TimeoutError()
Exception raised in Job[8]: TimeoutError()
Evaluating: 100%|██████████| 40/40 [03:30<00:00,  5.26s/it]  


RAGAS EVALUATION RESULTS
{'faithfulness': 0.5625, 'answer_relevancy': 0.7231, 'context_recall': 1.0000, 'factual_correctness(mode=f1)': 0.6667, 'context_precision': 0.9479}


: 